In [2]:
!pip install -q delta-spark pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 657.7 kB/s eta 0:00:00


In [3]:
import delta
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Part_3_Delta_lake") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(delta.__version__)
print("Spark session ready")


4.2.0
Spark session ready


In [ ]:
spark.sql(" show tables").show()

In [5]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]

df = spark.createDataFrame(data, columns)
df.show()
df.write.format("delta").mode("overwrite").save("/tmp/patient_visits_delta")

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
+--------+------------+---------+-----------+----------------+-----------+



In [5]:
spark.sql("""
CREATE TABLE patient_visits
USING DELTA
LOCATION '/tmp/patient_visits_delta'
""")

DataFrame[]

In [6]:
spark.sql("""
INSERT INTO patient_visits VALUES
(109,'Rahul Verma','Mumbai','Cardiology',6000,2)
""")

DataFrame[]

In [7]:
spark.sql("""
update patient_visits set consultation_fee = 3500
where visit_id = 102 """)

DataFrame[num_affected_rows: bigint]

In [8]:
spark.sql("""
delete from patient_visits where tests_count = 1
""")

DataFrame[num_affected_rows: bigint]

In [9]:
incoming_data = [
(102,"Sneha Kapoor","Delhi","Orthopedics",4000,3),
(110,"Divya Menon","Kochi","Cardiology",5000,1)
]

incoming_df=spark.createDataFrame(incoming_data, columns)
incoming_df.createOrReplaceTempView("incoming_visits")

In [11]:
spark.sql("""
merge into patient_visits as t
using incoming_visits as s
on t.visit_id = s.visit_id
when matched then update set *
when not matched then insert *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [12]:
spark.sql(" select * from patient_visits").show()

+--------+------------+---------+-----------+----------------+-----------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|
+--------+------------+---------+-----------+----------------+-----------+
|     102|Sneha Kapoor|    Delhi|Orthopedics|            4000|          3|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|
|     110| Divya Menon|    Kochi| Cardiology|            5000|          1|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|
|     109| Rahul Verma|   Mumbai| Cardiology|            6000|          2|
+--------+------------+---------+-----------+----------------+-----------+

